### Consolidated Setup: Data Loading, Preprocessing, Feature Engineering, and Model Training

This section consolidates all the necessary steps for data preparation, feature engineering, product categorization, and model training (both for dimensions and weight). This replaces multiple fragmented cells and ensures consistency across the notebook.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
print("--- Starting Consolidated Data Loading and Initial Preprocessing ---")

# --- 1. Load the dataset ---
file_path = '/content/cubagem_40k_amazon.csv'
df_full = pd.read_csv(file_path)

# --- 2. Data Preprocessing and Initial Renaming/Log Transformation ---

# Rename columns for easier access and consistency
df_full.rename(columns={
    'Peso (g)': 'weight_g',
    'Comprimento (cm)': 'length_cm',
    'Largura (cm)': 'width_cm',
    'Altura (cm)': 'height_cm',
    'price_numeric': 'price' # Rename price_numeric for consistent feature naming
}, inplace=True)

# Apply logarithmic transformation to target variables
df_full['log_weight_g'] = np.log1p(df_full['weight_g'])
df_full['log_length_cm'] = np.log1p(df_full['length_cm'])
df_full['log_width_cm'] = np.log1p(df_full['width_cm'])
df_full['log_height_cm'] = np.log1p(df_full['height_cm'])

# Define essential columns for initial NaN removal.
# Explicitly excluding 'volume_cm3' and 'density_g_cm3' as per user request.
essential_cols_for_dropna = ['weight_g', 'length_cm', 'width_cm', 'height_cm', 'price',
                           'dim_max_cm', 'dim_min_cm', 'dim_ratio']

initial_rows_full = len(df_full)
df_full.dropna(subset=essential_cols_for_dropna, inplace=True)
print(f"Dropped {initial_rows_full - len(df_full)} rows with missing values in essential columns.")

print("--- Initial Preprocessing Complete ---")

In [ ]:
df = pd.read_csv("../data/cubagem_40k_amazon.csv")

In [ ]:
# não elimine esse código
print("--- Starting Feature Engineering ---")

# --- 3. Feature Engineering (excluding volume and density) ---

df_full['total_surface_area_cm2'] = 2 * (df_full['length_cm'] * df_full['width_cm'] +
                                       df_full['length_cm'] * df_full['height_cm'] +
                                       df_full['width_cm'] * df_full['height_cm'])
df_full['ratio_min_to_max_dim'] = df_full['dim_min_cm'] / df_full['dim_max_cm']
df_full['mean_three_dims_cm'] = (df_full['length_cm'] + df_full['width_cm'] + df_full['height_cm']) / 3
df_full['length_width_aspect_ratio'] = df_full['length_cm'] / df_full['width_cm']
df_full['length_height_aspect_ratio'] = df_full['length_cm'] / df_full['height_cm']
df_full['width_height_aspect_ratio'] = df_full['width_cm'] / df_full['height_cm']

df_full['geometric_mean_dims'] = np.cbrt(df_full['length_cm'] * df_full['width_cm'] * df_full['height_cm'])
global numerical_features_for_dimensions
df_full['sphericity'] = np.where(
    df_full['mean_three_dims_cm'] > 0,
    df_full['geometric_mean_dims'] / df_full['mean_three_dims_cm'],
    0
)

# Define the final numerical features list for dimension prediction (explicitly exclude volume/density)
numerical_features_for_dimensions = [
    'price', 'dim_max_cm', 'dim_min_cm', 'dim_ratio',
    'total_surface_area_cm2', 'ratio_min_to_max_dim',
    'mean_three_dims_cm', 'length_width_aspect_ratio',
    'length_height_aspect_ratio', 'width_height_aspect_ratio', 'sphericity'
]

print("--- Feature Engineering Complete ---")

--- Starting Feature Engineering ---
--- Feature Engineering Complete ---


In [ ]:
# não elimine esse código
print("--- Starting Product Categorization ---")

# --- 4. Product Categorization (Small, Medium, Large) ---

# Use 'mean_three_dims_cm' for categorization
# Define quantiles to create roughly equal-sized categories
global quantiles
quantiles = df_full['mean_three_dims_cm'].quantile([0.33, 0.66])

global categorize_product
def categorize_product(mean_dim):
    if mean_dim <= quantiles[0.33]:
        return 'small'
    elif mean_dim <= quantiles[0.66]:
        return 'medium'
    else:
        return 'large'

df_full['size_category'] = df_full['mean_three_dims_cm'].apply(categorize_product)

print(f"Product categorization based on 'mean_three_dims_cm':")
display(df_full['size_category'].value_counts())

print("--- Product Categorization Complete ---")

--- Starting Product Categorization ---
Product categorization based on 'mean_three_dims_cm':


,count
size_category,
large,6953
small,6768
medium,6738


--- Product Categorization Complete ---


In [ ]:
# não elimine esse código
print("--- Starting Dimension Models Training (Categorized) ---")

# Define target columns for dimensions
target_cols_renamed_dimensions = ['log_length_cm', 'log_width_cm', 'log_height_cm']

# Store assumed optimized parameters for each dimension (from previous analysis)
assumed_optimized_params = {
    'log_length_cm': {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0},
    'log_width_cm': {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0},
    'log_height_cm': {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0}
}

global all_category_results
all_category_results = {}

for category in ['small', 'medium', 'large']:
    print(f"\n--- Training and Evaluating Models for '{category.upper()}' Category ---")

    df_category = df_full[df_full['size_category'] == category].copy()

    if df_category.empty:
        print(f"No data for '{category}' category. Skipping.")
        continue

    X_category = df_category[numerical_features_for_dimensions]
    Y_category = df_category[target_cols_renamed_dimensions]

    # Split data into training and testing sets for the current category
    X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
        X_category, Y_category, test_size=0.2, random_state=42
    )

    category_models = {}
    category_metrics_log_scale = {}
    category_relative_mae_original_scale = {}

    for col in target_cols_renamed_dimensions:
        print(f"  Training model for: {col} in {category} category")

        params = assumed_optimized_params.get(col, {}) # Get best params for the dimension

        model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, **params)
        model.fit(X_train_cat, y_train_cat[col])
        category_models[col] = model

        y_pred_log = model.predict(X_test_cat)
        mae_log = mean_absolute_error(y_test_cat[col], y_pred_log)
        rmse_log = np.sqrt(mean_squared_error(y_test_cat[col], y_pred_log))
        category_metrics_log_scale[col] = {'MAE_log': mae_log, 'RMSE_log': rmse_log}

        # Calculate Relative MAE on Original Scale
        y_pred_original = np.expm1(y_pred_log)
        y_actual_original = np.expm1(y_test_cat[col])

        mae_original = mean_absolute_error(y_actual_original, y_pred_original)
        avg_actual_original = y_actual_original.mean()

        relative_mae_percent = (mae_original / avg_actual_original) * 100 if avg_actual_original > 0 else np.nan

        original_col_name = col.replace('log_', '')
        category_relative_mae_original_scale[original_col_name] = {
            'MAE_original_scale': mae_original,
            'Avg_Actual_original_scale': avg_actual_original,
            'Relative_MAE_%': relative_mae_percent
        }

        print(f"    MAE (log scale): {mae_log:.4f}, RMSE (log scale): {rmse_log:.4f}")
        print(f"    Relative MAE ({original_col_name.capitalize()} original scale): {relative_mae_percent:.2f}%")

    all_category_results[category] = {
        'models': category_models,
        'metrics_log_scale': category_metrics_log_scale,
        'relative_mae_original_scale': category_relative_mae_original_scale
    }

print("--- Dimension Models Training Complete ---")

--- Starting Dimension Models Training (Categorized) ---

--- Training and Evaluating Models for 'SMALL' Category ---
  Training model for: log_length_cm in small category
    MAE (log scale): 0.0038, RMSE (log scale): 0.0080
    Relative MAE (Length_cm original scale): 0.38%
  Training model for: log_width_cm in small category
    MAE (log scale): 0.0134, RMSE (log scale): 0.0197
    Relative MAE (Width_cm original scale): 1.40%
  Training model for: log_height_cm in small category
    MAE (log scale): 0.0028, RMSE (log scale): 0.0038
    Relative MAE (Height_cm original scale): 0.40%

--- Training and Evaluating Models for 'MEDIUM' Category ---
  Training model for: log_length_cm in medium category
    MAE (log scale): 0.0021, RMSE (log scale): 0.0042
    Relative MAE (Length_cm original scale): 0.24%
  Training model for: log_width_cm in medium category
    MAE (log scale): 0.0104, RMSE (log scale): 0.0155
    Relative MAE (Width_cm original scale): 1.01%
  Training model for: log_h

In [ ]:
# não elimine esse código
print("--- Defining Utility Functions ---")

# --- Correios Box Recommendation Function ---
# Define example Correios box sizes (Length, Width, Height) in cm
correios_boxes = [
    {'name': 'Caixa Pequena', 'length_cm': 16, 'width_cm': 11, 'height_cm': 6, 'max_weight_g': 1000},
    {'name': 'Caixa Média', 'length_cm': 27, 'width_cm': 18, 'height_cm': 9, 'max_weight_g': 3000},
    {'name': 'Caixa Grande', 'length_cm': 36, 'width_cm': 27, 'height_cm': 12, 'max_weight_g': 5000},
    {'name': 'Caixa Extra Grande', 'length_cm': 45, 'width_cm': 30, 'height_cm': 20, 'max_weight_g': 10000}
]

def recommend_correios_box(predicted_dims: dict):
    prod_l = predicted_dims['length_cm']
    prod_w = predicted_dims['width_cm']
    prod_h = predicted_dims['height_cm']
    prod_weight = predicted_dims['weight_g']

    suitable_boxes = []

    for box in correios_boxes:
        box_dims = sorted([box['length_cm'], box['width_cm'], box['height_cm']])
        prod_dims = sorted([prod_l, prod_w, prod_h])

        if (prod_dims[0] <= box_dims[0] and
            prod_dims[1] <= box_dims[1] and
            prod_dims[2] <= box_dims[2] and
            prod_weight <= box['max_weight_g']):
            suitable_boxes.append(box)

    if not suitable_boxes:
        return "No suitable Correios box found from the defined list for these dimensions/weight."
    else:
        suitable_boxes.sort(key=lambda b: b['length_cm'] * b['width_cm'] * b['height_cm'])
        return suitable_boxes[0]['name']

# --- Product Dimension and Weight Prediction Function ---
def predict_product_attributes_final(numerical_inputs: dict):
    # Ensure global variables are accessible and defined.
    if 'numerical_features_for_dimensions' not in globals() or \
       'all_category_results' not in globals() or \
       'quantiles' not in globals() or \
       'categorize_product' not in globals():
        print("Error: Essential global variables for dimension prediction (numerical_features_for_dimensions, all_category_results, quantiles, categorize_product) are not defined. Ensure all setup cells have been run.")
        return {}

    input_df = pd.DataFrame([numerical_inputs])

    # --- 1. Determine Product Category ---
    if 'mean_three_dims_cm' not in input_df.columns or input_df['mean_three_dims_cm'].isnull().any():
        print("Error: 'mean_three_dims_cm' is missing or NaN in numerical_inputs. Cannot categorize product for dimension prediction.")
        return {}

    product_mean_dim = input_df['mean_three_dims_cm'].iloc[0]
    category = categorize_product(product_mean_dim)

    # --- 2. Select Category-Specific Models for Dimensions ---
    if category not in all_category_results or 'models' not in all_category_results[category]:
        print(f"Error: Models for category '{category}' not found in 'all_category_results'. Ensure models were trained for all categories.")
        return {}

    category_specific_models = all_category_results[category]['models']

    # --- 3. Prepare Input Features for Dimension Prediction ---
    try:
        input_features_for_model_dims = input_df[numerical_features_for_dimensions]
    except KeyError as e:
        print(f"Error: Missing feature(s) in numerical_inputs required by dimension models: {e}. Please ensure all features in 'numerical_features_for_dimensions' are provided.")
        return {}

    predicted_attributes = {}
    dimension_target_cols = ['log_length_cm', 'log_width_cm', 'log_height_cm']

    for col in dimension_target_cols:
        if col in category_specific_models:
            prediction_log = category_specific_models[col].predict(input_features_for_model_dims)[0]
            original_col_name = col.replace('log_', '')
            predicted_attributes[original_col_name] = np.expm1(prediction_log)
        else:
            print(f"Warning: Model for {col} not found in {category} category models. Skipping prediction for {original_col_name}.")

    # --- 4. Handle weight_g prediction ---
    if 'weight_g' in numerical_inputs: # If weight_g is explicitly provided in input, use it.
        predicted_attributes['weight_g'] = numerical_inputs['weight_g']
    elif 're_trained_optimized_models_new_fe' in globals() and \
         'log_weight_g' in re_trained_optimized_models_new_fe and \
         'numerical_features_full' in globals():
        try:
            weight_model = re_trained_optimized_models_new_fe['log_weight_g']
            input_features_for_model_weight = input_df[numerical_features_full]
            weight_prediction_log = weight_model.predict(input_features_for_model_weight)[0]
            predicted_attributes['weight_g'] = np.expm1(weight_prediction_log)
        except KeyError as e:
            print(f"Warning: Missing feature(s) in numerical_inputs for weight prediction fallback: {e}. Skipping weight prediction.")
        except Exception as e:
            print(f"Warning: Error during weight prediction fallback: {e}. Skipping weight prediction.")
    else:
        print("Warning: 'weight_g' not provided in numerical_inputs and a suitable weight prediction model was not found for fallback.")

    return predicted_attributes

print("--- Utility Functions Defined ---")

--- Defining Utility Functions ---
--- Utility Functions Defined ---


### Demonstration of Prediction with Consolidated Models

This cell demonstrates how to use the `predict_product_attributes_final` function with the models and features trained in the consolidated setup. This addresses the previous `ValueError: feature_names mismatch` by ensuring all required features are correctly provided.

In [ ]:
# não elimine esse código
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# --- 1. Load the dataset ---
file_path = '/content/cubagem_40k_amazon.csv'
df_full = pd.read_csv(file_path)

print("Dataset loaded successfully.")

# --- 2. Data Preprocessing and Feature Engineering for Dimensions ---

# Rename columns for easier access and consistency
df_full.rename(columns={
    'Peso (g)': 'weight_g',
    'Comprimento (cm)': 'length_cm',
    'Largura (cm)': 'width_cm',
    'Altura (cm)': 'height_cm',
    'price_numeric': 'price' # Rename price_numeric for consistent feature naming
}, inplace=True)

# Apply logarithmic transformation to target variables
df_full['log_weight_g'] = np.log1p(df_full['weight_g'])
df_full['log_length_cm'] = np.log1p(df_full['length_cm'])
df_full['log_width_cm'] = np.log1p(df_full['width_cm'])
df_full['log_height_cm'] = np.log1p(df_full['height_cm'])

# Define all target columns (log-transformed) - Focusing only on dimensions here
target_cols_renamed = ['log_length_cm', 'log_width_cm', 'log_height_cm']

# Ensure that columns used for feature engineering are present and handle missing values.
# The engineered features rely on 'length_cm', 'width_cm', 'height_cm', 'dim_max_cm', 'dim_min_cm'.
# 'price' is also a critical feature.
essential_cols = ['weight_g', 'length_cm', 'width_cm', 'height_cm', 'price',
                  'dim_max_cm', 'dim_min_cm', 'dim_ratio']

initial_rows_full = len(df_full)
df_full.dropna(subset=essential_cols, inplace=True)
print(f"Dropped {initial_rows_full - len(df_full)} rows with missing values in essential columns.")

# Engineer new dimension-based features (from cell b6569c95)
df_full['total_surface_area_cm2'] = 2 * (df_full['length_cm'] * df_full['width_cm'] +
                                       df_full['length_cm'] * df_full['height_cm'] +
                                       df_full['width_cm'] * df_full['height_cm'])
df_full['ratio_min_to_max_dim'] = df_full['dim_min_cm'] / df_full['dim_max_cm']
df_full['mean_three_dims_cm'] = (df_full['length_cm'] + df_full['width_cm'] + df_full['height_cm']) / 3
df_full['length_width_aspect_ratio'] = df_full['length_cm'] / df_full['width_cm']
df_full['length_height_aspect_ratio'] = df_full['length_cm'] / df_full['height_cm']
df_full['width_height_aspect_ratio'] = df_full['width_cm'] / df_full['height_cm']

# Add sphericity feature
df_full['geometric_mean_dims'] = np.cbrt(df_full['length_cm'] * df_full['width_cm'] * df_full['height_cm'])
df_full['sphericity'] = np.where(
    df_full['mean_three_dims_cm'] > 0,
    df_full['geometric_mean_dims'] / df_full['mean_three_dims_cm'],
    0
)

# Define the final numerical features list for dimension prediction
numerical_features_for_dimensions = ['price', 'dim_max_cm', 'dim_min_cm', 'dim_ratio',
                                     'total_surface_area_cm2', 'ratio_min_to_max_dim',
                                     'mean_three_dims_cm', 'length_width_aspect_ratio',
                                     'length_height_aspect_ratio', 'width_height_aspect_ratio', 'sphericity']

# Prepare X (features) and Y (targets) for training
X_full = df_full[numerical_features_for_dimensions]
Y_full = df_full[target_cols_renamed]

print(f"X_full shape (features): {X_full.shape}")
print(f"Y_full shape (targets): {Y_full.shape}")
print("Data preprocessing and feature engineering complete.")

# --- 3. Split Data into Training and Testing Sets ---
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_full, Y_full, test_size=0.2, random_state=42)
print(f"Training set size: {len(X_train_full)} samples")
print(f"Test set size: {len(X_test_full)} samples")
print("Data split into training and testing sets.")

# --- 4. Re-train Optimized XGBoost Models for Dimension Targets ---
print("\n--- Re-training Optimized Models for Dimension Targets ---")

# Define best hyperparameters for each target based on previous optimization (from context)
assumed_optimized_params = {
    # Removed log_weight_g as this cell is for dimension prediction
    'log_length_cm': {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0},
    'log_width_cm': {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0},
    'log_height_cm': {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0}
}

re_trained_optimized_models_final = {}
re_evaluated_metrics_final = {}

# Only focus on dimension targets for re-training and evaluation as requested
dimension_targets = ['log_length_cm', 'log_width_cm', 'log_height_cm']

for col in dimension_targets:
    print(f"\nRe-training and evaluating model for: {col}")

    # Get the best parameters for the current target
    params = assumed_optimized_params.get(col, {}) # Use the assumed parameters

    # Instantiate XGBoost Regressor with optimized parameters
    model_re_trained = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, **params)

    # Train on the full training set
    model_re_trained.fit(X_train_full, y_train_full[col])
    re_trained_optimized_models_final[col] = model_re_trained

    # Make predictions on the full test set
    y_pred_full = model_re_trained.predict(X_test_full)

    # Evaluate the model (MAE and RMSE on log scale)
    mae_full = mean_absolute_error(y_test_full[col], y_pred_full)
    rmse_full = np.sqrt(mean_squared_error(y_test_full[col], y_pred_full))
    re_evaluated_metrics_final[col] = {'MAE': mae_full, 'RMSE': rmse_full}
    print(f"  MAE on log scale for {col}: {mae_full:.4f}")
    print(f"  RMSE on log scale for {col}: {rmse_full:.4f}")

print("\n--- Re-training and Evaluation on Log Scale Complete ---")
print("Re-evaluated Metrics (on log scale):")
display(pd.DataFrame(re_evaluated_metrics_final))

# --- 5. Calculate Relative MAE on Original Scale for Dimension Targets ---
print("\n--- Calculating Relative MAE on Original Scale (Dimension Targets) ---")

relative_mae_results_final = {}

for col in dimension_targets:
    model = re_trained_optimized_models_final[col]

    # Make predictions on the full test set (log-transformed)
    y_pred_log = model.predict(X_test_full)
    # Get actual values from the full test set (log-transformed)
    y_actual_log = y_test_full[col]

    # Inverse transform predictions and actual values to original scale
    y_pred_original = np.expm1(y_pred_log)
    y_actual_original = np.expm1(y_actual_log)

    # Calculate MAE on the original scale
    mae_original_scale = mean_absolute_error(y_actual_original, y_pred_original)

    # Calculate the average actual value on the original scale
    avg_actual_original = y_actual_original.mean()

    # Calculate relative MAE
    if avg_actual_original > 0:
        relative_mae = (mae_original_scale / avg_actual_original) * 100
    else:
        relative_mae = np.nan # Avoid division by zero

    original_col_name = col.replace('log_', '') # Get original column name (e.g., length_cm)
    relative_mae_results_final[original_col_name] = {
        'MAE_original_scale': mae_original_scale,
        'Avg_Actual_original_scale': avg_actual_original,
        'Relative_MAE_%': relative_mae
    }

# Display the results
relative_mae_df_final = pd.DataFrame(relative_mae_results_final).T
print("\nRelative Mean Absolute Error (MAE) on Original Scale (Dimension Targets):")
display(relative_mae_df_final)

# Check against the 5% target
print("\n--- Checking Relative MAE against 5% target (Dimension Targets) ---")
for product, metrics in relative_mae_results_final.items():
    if metrics['Relative_MAE_%'] <= 2:
        print(f"  {product.capitalize()} Relative MAE ({metrics['Relative_MAE_%']:.2f}%) is within the 2% target.")
    elif metrics['Relative_MAE_%'] <= 5:
        print(f"  {product.capitalize()} Relative MAE ({metrics['Relative_MAE_%']:.2f}%) is within the 5% target (but not 2%).")
    else:
        print(f"  WARNING: {product.capitalize()} Relative MAE ({metrics['Relative_MAE_%']:.2f}%) EXCEEDS the 5% target.")

Dataset loaded successfully.
Dropped 19541 rows with missing values in essential columns.
X_full shape (features): (20459, 11)
Y_full shape (targets): (20459, 3)
Data preprocessing and feature engineering complete.
Training set size: 16367 samples
Test set size: 4092 samples
Data split into training and testing sets.

--- Re-training Optimized Models for Dimension Targets ---

Re-training and evaluating model for: log_length_cm
  MAE on log scale for log_length_cm: 0.0066
  RMSE on log scale for log_length_cm: 0.0122

Re-training and evaluating model for: log_width_cm
  MAE on log scale for log_width_cm: 0.0169
  RMSE on log scale for log_width_cm: 0.0235

Re-training and evaluating model for: log_height_cm
  MAE on log scale for log_height_cm: 0.0069
  RMSE on log scale for log_height_cm: 0.0164

--- Re-training and Evaluation on Log Scale Complete ---
Re-evaluated Metrics (on log scale):


,log_length_cm,log_width_cm,log_height_cm
MAE,0.006610,0.016926,0.00687
RMSE,0.012155,0.023536,0.01636



--- Calculating Relative MAE on Original Scale (Dimension Targets) ---

Relative Mean Absolute Error (MAE) on Original Scale (Dimension Targets):


,MAE_original_scale,Avg_Actual_original_scale,Relative_MAE_%
length_cm,0.241358,27.284382,0.884603
width_cm,0.335599,17.005374,1.973489
height_cm,0.119575,7.230963,1.653655



--- Checking Relative MAE against 5% target (Dimension Targets) ---
  Length_cm Relative MAE (0.88%) is within the 2% target.
  Width_cm Relative MAE (1.97%) is within the 2% target.
  Height_cm Relative MAE (1.65%) is within the 2% target.


In [ ]:
# não elimine esse código
import pandas as pd
import numpy as np

def predict_product_attributes_final(numerical_inputs: dict):
    # Ensure global variables are accessible and defined.
    # These are expected to be set by running cell ac2ec59d and other relevant cells.
    if 'numerical_features_for_dimensions' not in globals() or \
       'all_category_results' not in globals() or \
       'quantiles' not in globals() or \
       'categorize_product' not in globals():
        print("Error: Essential global variables for dimension prediction (numerical_features_for_dimensions, all_category_results, quantiles, categorize_product) are not defined. Ensure all setup cells have been run.")
        return {}

    # Create a DataFrame from the input for feature consistency and easier processing.
    input_df = pd.DataFrame([numerical_inputs])

    # --- 1. Determine Product Category ---
    # The 'mean_three_dims_cm' is crucial for categorization. It must be present in the input.
    if 'mean_three_dims_cm' not in input_df.columns or input_df['mean_three_dims_cm'].isnull().any():
        print("Error: 'mean_three_dims_cm' is missing or NaN in numerical_inputs. Cannot categorize product for dimension prediction.")
        return {}

    product_mean_dim = input_df['mean_three_dims_cm'].iloc[0]
    category = categorize_product(product_mean_dim)

    # --- 2. Select Category-Specific Models for Dimensions ---
    if category not in all_category_results or 'models' not in all_category_results[category]:
        print(f"Error: Models for category '{category}' not found in 'all_category_results'. Ensure models were trained for all categories.")
        return {}

    category_specific_models = all_category_results[category]['models']

    # --- 3. Prepare Input Features for Dimension Prediction ---
    # Filter and reorder the input features to match the training data's feature set.
    try:
        input_features_for_model = input_df[numerical_features_for_dimensions]
    except KeyError as e:
        print(f"Error: Missing feature(s) in numerical_inputs required by dimension models: {e}. Please ensure all features in 'numerical_features_for_dimensions' are provided.")
        return {}

    predicted_attributes = {}
    dimension_target_cols = ['log_length_cm', 'log_width_cm', 'log_height_cm']

    for col in dimension_target_cols:
        if col in category_specific_models:
            prediction_log = category_specific_models[col].predict(input_features_for_model)[0]
            original_col_name = col.replace('log_', '')
            predicted_attributes[original_col_name] = np.expm1(prediction_log)
        else:
            print(f"Warning: Model for {col} not found in {category} category models. Skipping prediction for {original_col_name}.")

    # --- 4. Handle weight_g for box recommendation ---
    # If weight_g is provided in numerical_inputs, it will be used for box recommendation.
    # Automatic weight prediction has been removed as per user's request.
    if 'weight_g' in numerical_inputs:
        predicted_attributes['weight_g'] = numerical_inputs['weight_g']
    else:
        # If weight_g is not provided, and no automatic prediction is desired, just print a warning.
        print("Warning: 'weight_g' not provided in numerical_inputs. Weight prediction is not performed as per user's request.")

    return predicted_attributes

print("Final prediction function updated to use models and features for category-specific dimension predictions only. Weight prediction logic removed.")

Final prediction function updated to use models and features for category-specific dimension predictions only. Weight prediction logic removed.


In [ ]:
# não elimine esse código
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. Load the dataset ---
file_path = '/content/cubagem_40k_amazon.csv'
df_full = pd.read_csv(file_path)

print("Dataset loaded successfully.")

# --- 2. Data Preprocessing and Initial Renaming/Log Transformation ---

# Rename columns for easier access and consistency
df_full.rename(columns={
    'Peso (g)': 'weight_g',
    'Comprimento (cm)': 'length_cm',
    'Largura (cm)': 'width_cm',
    'Altura (cm)': 'height_cm',
    'price_numeric': 'price' # Rename price_numeric for consistent feature naming
}, inplace=True)

# Apply logarithmic transformation to target variables
df_full['log_weight_g'] = np.log1p(df_full['weight_g'])
df_full['log_length_cm'] = np.log1p(df_full['length_cm'])
df_full['log_width_cm'] = np.log1p(df_full['width_cm'])
df_full['log_height_cm'] = np.log1p(df_full['height_cm'])

# Define all target columns (log-transformed)
target_cols_renamed = ['log_length_cm', 'log_width_cm', 'log_height_cm'] # Focus only on dimensions

# Define essential columns for initial NaN removal. Exclude 'volume_cm3' and 'density_g_cm3'.
essential_cols_for_dropna = ['weight_g', 'length_cm', 'width_cm', 'height_cm', 'price',
                           'dim_max_cm', 'dim_min_cm', 'dim_ratio']

initial_rows_full = len(df_full)
df_full.dropna(subset=essential_cols_for_dropna, inplace=True)
print(f"Dropped {initial_rows_full - len(df_full)} rows with missing values in essential columns.")

# --- 3. Feature Engineering (excluding volume and density) ---

df_full['total_surface_area_cm2'] = 2 * (df_full['length_cm'] * df_full['width_cm'] +
                                       df_full['length_cm'] * df_full['height_cm'] +
                                       df_full['width_cm'] * df_full['height_cm'])
df_full['ratio_min_to_max_dim'] = df_full['dim_min_cm'] / df_full['dim_max_cm']
df_full['mean_three_dims_cm'] = (df_full['length_cm'] + df_full['width_cm'] + df_full['height_cm']) / 3
df_full['length_width_aspect_ratio'] = df_full['length_cm'] / df_full['width_cm']
df_full['length_height_aspect_ratio'] = df_full['length_cm'] / df_full['height_cm']
df_full['width_height_aspect_ratio'] = df_full['width_cm'] / df_full['height_cm']

df_full['geometric_mean_dims'] = np.cbrt(df_full['length_cm'] * df_full['width_cm'] * df_full['height_cm'])
df_full['sphericity'] = np.where(
    df_full['mean_three_dims_cm'] > 0,
    df_full['geometric_mean_dims'] / df_full['mean_three_dims_cm'],
    0
)

# Define the final numerical features list for dimension prediction (explicitly exclude volume/density)
global numerical_features_for_dimensions # Make it global
numerical_features_for_dimensions = [
    'price', 'dim_max_cm', 'dim_min_cm', 'dim_ratio',
    'total_surface_area_cm2', 'ratio_min_to_max_dim',
    'mean_three_dims_cm', 'length_width_aspect_ratio',
    'length_height_aspect_ratio', 'width_height_aspect_ratio', 'sphericity'
]

print("Feature engineering complete.")

# --- 4. Product Categorization (Small, Medium, Large) ---

# Use 'mean_three_dims_cm' for categorization
# Define quantiles to create roughly equal-sized categories
global quantiles # Make quantiles global
quantiles = df_full['mean_three_dims_cm'].quantile([0.33, 0.66])

global categorize_product # Make categorize_product global
def categorize_product(mean_dim):
    if mean_dim <= quantiles[0.33]:
        return 'small'
    elif mean_dim <= quantiles[0.66]:
        return 'medium'
    else:
        return 'large'

df_full['size_category'] = df_full['mean_three_dims_cm'].apply(categorize_product)

print(f"Product categorization based on 'mean_three_dims_cm':")
display(df_full['size_category'].value_counts())

# Store assumed optimized parameters for each dimension from previous analysis (cell 95910cb6)
assumed_optimized_params = {
    'log_length_cm': {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0},
    'log_width_cm': {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0},
    'log_height_cm': {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0}
}

# --- 5. Model Training and Evaluation by Category ---

global all_category_results # Make it global
all_category_results = {}

for category in ['small', 'medium', 'large']:
    print(f"\n--- Training and Evaluating Models for '{category.upper()}' Category ---")

    df_category = df_full[df_full['size_category'] == category].copy()

    if df_category.empty:
        print(f"No data for '{category}' category. Skipping.")
        continue

    X_category = df_category[numerical_features_for_dimensions]
    Y_category = df_category[target_cols_renamed]

    # Split data into training and testing sets for the current category
    X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
        X_category, Y_category, test_size=0.2, random_state=42
    )

    category_models = {}
    category_metrics_log_scale = {}
    category_relative_mae_original_scale = {}

    for col in target_cols_renamed:
        print(f"  Training model for: {col} in {category} category")

        params = assumed_optimized_params.get(col, {}) # Get best params for the dimension

        model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, **params)
        model.fit(X_train_cat, y_train_cat[col])
        category_models[col] = model

        y_pred_log = model.predict(X_test_cat)
        mae_log = mean_absolute_error(y_test_cat[col], y_pred_log)
        rmse_log = np.sqrt(mean_squared_error(y_test_cat[col], y_pred_log))
        category_metrics_log_scale[col] = {'MAE_log': mae_log, 'RMSE_log': rmse_log}

        # Calculate Relative MAE on Original Scale
        y_pred_original = np.expm1(y_pred_log)
        y_actual_original = np.expm1(y_test_cat[col])

        mae_original = mean_absolute_error(y_actual_original, y_pred_original)
        avg_actual_original = y_actual_original.mean()

        relative_mae_percent = (mae_original / avg_actual_original) * 100 if avg_actual_original > 0 else np.nan

        original_col_name = col.replace('log_', '')
        category_relative_mae_original_scale[original_col_name] = {
            'MAE_original_scale': mae_original,
            'Avg_Actual_original_scale': avg_actual_original,
            'Relative_MAE_%': relative_mae_percent
        }

        print(f"    MAE (log scale): {mae_log:.4f}, RMSE (log scale): {rmse_log:.4f}")
        print(f"    Relative MAE ({original_col_name.capitalize()} original scale): {relative_mae_percent:.2f}%")

    all_category_results[category] = {
        'models': category_models,
        'metrics_log_scale': category_metrics_log_scale,
        'relative_mae_original_scale': category_relative_mae_original_scale
    }

    print(f"\n--- Evaluation Metrics for '{category.upper()}' Category (Log Scale) ---")
    display(pd.DataFrame(category_metrics_log_scale).T)
    print(f"\n--- Relative MAE for '{category.upper()}' Category (Original Scale) ---")
    display(pd.DataFrame(category_relative_mae_original_scale).T)

print("\n--- Consolidated Training and Evaluation Complete ---")
print("Summary of Relative MAE on Original Scale by Category:")
for category, results in all_category_results.items():
    print(f"\nCategory: {category.upper()}")
    display(pd.DataFrame(results['relative_mae_original_scale']).T)

print("\n--- Checking Relative MAE against 5% target (All Categories) ---")
for category, results in all_category_results.items():
    print(f"\nCategory: {category.upper()}")
    for product, metrics in results['relative_mae_original_scale'].items():
        if metrics['Relative_MAE_%'] <= 5:
            print(f"  {product.capitalize()} Relative MAE ({metrics['Relative_MAE_%']:.2f}%) is within the 5% target.")
        else:
            print(f"  WARNING: {product.capitalize()} Relative MAE ({metrics['Relative_MAE_%']:.2f}%) EXCEEDS the 5% target.")


Dataset loaded successfully.
Dropped 19541 rows with missing values in essential columns.
Feature engineering complete.
Product categorization based on 'mean_three_dims_cm':


,count
size_category,
large,6953
small,6768
medium,6738



--- Training and Evaluating Models for 'SMALL' Category ---
  Training model for: log_length_cm in small category
    MAE (log scale): 0.0038, RMSE (log scale): 0.0080
    Relative MAE (Length_cm original scale): 0.38%
  Training model for: log_width_cm in small category
    MAE (log scale): 0.0134, RMSE (log scale): 0.0197
    Relative MAE (Width_cm original scale): 1.40%
  Training model for: log_height_cm in small category
    MAE (log scale): 0.0028, RMSE (log scale): 0.0038
    Relative MAE (Height_cm original scale): 0.40%

--- Evaluation Metrics for 'SMALL' Category (Log Scale) ---


,MAE_log,RMSE_log
log_length_cm,0.003784,0.007956
log_width_cm,0.013445,0.019749
log_height_cm,0.002829,0.003773



--- Relative MAE for 'SMALL' Category (Original Scale) ---


...


Category: MEDIUM


,MAE_original_scale,Avg_Actual_original_scale,Relative_MAE_%
length_cm,0.053079,22.431818,0.236622
width_cm,0.145573,14.389955,1.011627
height_cm,0.032359,5.679607,0.569740



Category: LARGE


,MAE_original_scale,Avg_Actual_original_scale,Relative_MAE_%
length_cm,0.285101,47.348045,0.602138
width_cm,0.555392,29.097671,1.908716
height_cm,0.153377,11.533055,1.329887



--- Checking Relative MAE against 5% target (All Categories) ---

Category: SMALL
  Length_cm Relative MAE (0.38%) is within the 5% target.
  Width_cm Relative MAE (1.40%) is within the 5% target.
  Height_cm Relative MAE (0.40%) is within the 5% target.

Category: MEDIUM
  Length_cm Relative MAE (0.24%) is within the 5% target.
  Width_cm Relative MAE (1.01%) is within the 5% target.
  Height_cm Relative MAE (0.57%) is within the 5% target.

Category: LARGE
  Length_cm Relative MAE (0.60%) is within the 5% target.
  Width_cm Relative MAE (1.91%) is within the 5% target.
  Height_cm Relative MAE (1.33%) is within the 5% target.


In [ ]:
# não elimine esse código
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("\n--- Re-training Optimized Models on Full Training Data (with newly engineered sphericity feature) ---")
# Ensure target_cols_renamed is correctly defined for this cell's execution
# This list now EXCLUDES log_weight_g as per user request.
global target_cols_renamed # Ensure target_cols_renamed is global
target_cols_renamed = ['log_length_cm', 'log_width_cm', 'log_height_cm'] # Removed log_weight_g
print(f"Target columns for re-training: {target_cols_renamed}")

global re_trained_optimized_models_sphericity_fe
re_trained_optimized_models_sphericity_fe = {}

global re_evaluated_metrics_sphericity_fe
re_evaluated_metrics_sphericity_fe = {}

for col in target_cols_renamed:
    print(f"\nRe-training and evaluating model for: {col}")

    params = {}
    # Removed logic for 'log_weight_g' training as per user request.
    # Use assumed optimized parameters for dimensions (`ac2ec59d` or `95910cb6`)
    if 'assumed_optimized_params' in globals() and col in assumed_optimized_params:
        params = assumed_optimized_params[col]
    else:
        print(f"  Warning: Assumed optimized parameters for {col} not found in `assumed_optimized_params`. Using default XGBRegressor parameters.")

    # Instantiate XGBoost Regressor with optimized parameters
    model_re_trained = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, **params)

    # Train on the full training set with the new X_train_full (including sphericity)
    model_re_trained.fit(X_train_full, y_train_full[col])
    re_trained_optimized_models_sphericity_fe[col] = model_re_trained

    # Make predictions on the full test set with the new X_test_full (including sphericity)
    y_pred_full = model_re_trained.predict(X_test_full)

    # Evaluate the model
    mae_full = mean_absolute_error(y_test_full[col], y_pred_full)
    rmse_full = np.sqrt(mean_squared_error(y_test_full[col], y_pred_full))
    re_evaluated_metrics_sphericity_fe[col] = {'MAE': mae_full, 'RMSE': rmse_full}
    print(f"  MAE on larger test set for {col}: {mae_full:.2f}")
    print(f"  RMSE on larger test set for {col}: {rmse_full:.2f}")

print("\n--- Re-evaluation on Larger Test Set Complete (with newly engineered sphericity feature) ---")
print("Re-evaluated Metrics (on larger test set with newly engineered sphericity feature) :")
display(pd.DataFrame(re_evaluated_metrics_sphericity_fe))


--- Re-training Optimized Models on Full Training Data (with newly engineered sphericity feature) ---
Target columns for re-training: ['log_length_cm', 'log_width_cm', 'log_height_cm']

Re-training and evaluating model for: log_length_cm
  MAE on larger test set for log_length_cm: 0.01
  RMSE on larger test set for log_length_cm: 0.01

Re-training and evaluating model for: log_width_cm
  MAE on larger test set for log_width_cm: 0.02
  RMSE on larger test set for log_width_cm: 0.02

Re-training and evaluating model for: log_height_cm
  MAE on larger test set for log_height_cm: 0.01
  RMSE on larger test set for log_height_cm: 0.02

--- Re-evaluation on Larger Test Set Complete (with newly engineered sphericity feature) ---
Re-evaluated Metrics (on larger test set with newly engineered sphericity feature) :


,log_length_cm,log_width_cm,log_height_cm
MAE,0.006610,0.016926,0.00687
RMSE,0.012155,0.023536,0.01636


In [ ]:
# não elimine esse código
import pandas as pd
import numpy as np

print("Generating predictions for all items in df_full using models with sphericity feature...")

# --- Ensure df_full is present and fully prepared with all features ---

# Check if df_full exists globally and is a pandas DataFrame
if 'df_full' not in globals() or not isinstance(df_full, pd.DataFrame):
    print("Warning: df_full not found or not a DataFrame. Reloading from CSV.")
    file_path = '/content/cubagem_40k_amazon.csv'
    df_full = pd.read_csv(file_path)

# Ensure initial preprocessing (renaming, log transform, dropna)
# Check for a representative column from initial preprocessing, e.g., 'log_length_cm'
required_initial_cols = ['weight_g', 'length_cm', 'width_cm', 'height_cm', 'price',
                         'dim_max_cm', 'dim_min_cm', 'dim_ratio']
if not all(col in df_full.columns for col in required_initial_cols) or \
   'log_length_cm' not in df_full.columns:
    print("Warning: df_full not fully preprocessed. Applying initial preprocessing steps.")
    df_full.rename(columns={
        'Peso (g)': 'weight_g',
        'Comprimento (cm)': 'length_cm',
        'Largura (cm)': 'width_cm',
        'Altura (cm)': 'height_cm',
        'price_numeric': 'price' # Rename price_numeric for consistent feature naming
    }, inplace=True)

    # Apply logarithmic transformation to target variables
    df_full['log_weight_g'] = np.log1p(df_full['weight_g'])
    df_full['log_length_cm'] = np.log1p(df_full['length_cm'])
    df_full['log_width_cm'] = np.log1p(df_full['width_cm'])
    df_full['log_height_cm'] = np.log1p(df_full['height_cm'])

    # Define essential columns for initial NaN removal.
    initial_rows_before_dropna = len(df_full)
    df_full.dropna(subset=required_initial_cols, inplace=True)
    if len(df_full) < initial_rows_before_dropna:
        print(f"Dropped {initial_rows_before_dropna - len(df_full)} rows with missing values during preprocessing.")

# Ensure feature engineering
expected_engineered_features = [
    'total_surface_area_cm2', 'ratio_min_to_max_dim',
    'mean_three_dims_cm', 'length_width_aspect_ratio',
    'length_height_aspect_ratio', 'width_height_aspect_ratio', 'sphericity'
]
if not all(feature in df_full.columns for feature in expected_engineered_features):
    print("Warning: Missing engineered features in df_full. Applying feature engineering steps.")
    df_full['total_surface_area_cm2'] = 2 * (df_full['length_cm'] * df_full['width_cm'] +
                                           df_full['length_cm'] * df_full['height_cm'] +
                                           df_full['width_cm'] * df_full['height_cm'])
    df_full['ratio_min_to_max_dim'] = df_full['dim_min_cm'] / df_full['dim_max_cm']
    df_full['mean_three_dims_cm'] = (df_full['length_cm'] + df_full['width_cm'] + df_full['height_cm']) / 3
    df_full['length_width_aspect_ratio'] = df_full['length_cm'] / df_full['width_cm']
    df_full['length_height_aspect_ratio'] = df_full['length_cm'] / df_full['height_cm']
    df_full['width_height_aspect_ratio'] = df_full['width_cm'] / df_full['height_cm']

    df_full['geometric_mean_dims'] = np.cbrt(df_full['length_cm'] * df_full['width_cm'] * df_full['height_cm'])
    df_full['sphericity'] = np.where(
        df_full['mean_three_dims_cm'] > 0,
        df_full['geometric_mean_dims'] / df_full['mean_three_dims_cm'],
        0
    )

# Define numerical_features_for_dimensions globally and ensure it's up-to-date
global numerical_features_for_dimensions
numerical_features_for_dimensions = [
    'price', 'dim_max_cm', 'dim_min_cm', 'dim_ratio',
    'total_surface_area_cm2', 'ratio_min_to_max_dim',
    'mean_three_dims_cm', 'length_width_aspect_ratio',
    'length_height_aspect_ratio', 'width_height_aspect_ratio', 'sphericity'
]
# --- End of df_full preparation ---

# Prepare features for prediction for the entire df_full
X_full_predict = df_full[numerical_features_for_dimensions]

predictions_all_sphericity_fe = {}
actuals_all_sphericity_fe = {}
max_percentage_errors_sphericity_fe = {}
max_absolute_errors_sphericity_fe = {}

# Target columns for which we have models in re_trained_optimized_models_sphericity_fe
# These models were trained on the feature set `numerical_features_for_dimensions`
target_cols_to_predict = ['log_length_cm', 'log_width_cm', 'log_height_cm']

for col in target_cols_to_predict:
    # Get the model for the current target from re_trained_optimized_models_sphericity_fe
    # This variable is expected to be defined by a preceding cell like c07cc6ae
    if 're_trained_optimized_models_sphericity_fe' not in globals():
        print("Error: 're_trained_optimized_models_sphericity_fe' (trained models) not found. Please ensure relevant training cells have been run.")
        break # Exit loop if models are not available

    model = re_trained_optimized_models_sphericity_fe[col]

    # Make predictions on the full dataset (log-transformed)
    y_pred_log = model.predict(X_full_predict)

    # Get actual values from df_full (log-transformed)
    y_actual_log = df_full[col]

    # Inverse transform predictions and actual values to original scale
    y_pred_original = np.expm1(y_pred_log)
    y_actual_original = np.expm1(y_actual_log)

    original_col_name = col.replace('log_', '')
    predictions_all_sphericity_fe[original_col_name] = y_pred_original
    actuals_all_sphericity_fe[original_col_name] = y_actual_original

    # Calculate absolute error for each item
    absolute_error = np.abs(y_actual_original - y_pred_original)

    # Calculate percentage error for each item
    # Handle cases where actual_original is zero to avoid division by zero
    # Given physical dimensions and weight, values are expected to be > 0.
    percentage_error = np.abs((y_actual_original - y_pred_original) / y_actual_original) * 100

    # Replace any infinite values (which would occur if actual_original was 0 and y_pred_original was not) with NaN
    percentage_error = percentage_error.replace([np.inf, -np.inf], np.nan)

    # Find the maximum errors, ignoring NaN values
    max_absolute_errors_sphericity_fe[original_col_name] = np.nanmax(absolute_error) if not np.all(np.isnan(percentage_error)) else 0.0
    max_percentage_errors_sphericity_fe[original_col_name] = np.nanmax(percentage_error) if not np.all(np.isnan(percentage_error)) else 0.0

print("\nMaximum Errors (absolute and percentage) for all items with Sphericity Feature:")

# Create DataFrames for a more organized display
max_abs_errors_df = pd.DataFrame(max_absolute_errors_sphericity_fe.items(), columns=['Attribute', 'Max Absolute Error'])
max_perc_errors_df = pd.DataFrame(max_percentage_errors_sphericity_fe.items(), columns=['Attribute', 'Max Percentage Error (%)'])

print("\nMax Absolute Errors:")
display(max_abs_errors_df)

print("\nMax Percentage Errors:")
display(max_perc_errors_df)

Generating predictions for all items in df_full using models with sphericity feature...

Maximum Errors (absolute and percentage) for all items with Sphericity Feature:

Max Absolute Errors:


,Attribute,Max Absolute Error
0,length_cm,19.498574
1,width_cm,38.901472
2,height_cm,47.220714



Max Percentage Errors:


,Attribute,Max Percentage Error (%)
0,length_cm,76.665576
1,width_cm,32.124116
2,height_cm,40.822521


### Analyzing High Percentage Errors in Dimension Predictions

To understand the causes of the high percentage errors, especially the 76.67% for `length_cm`, we need to inspect the individual predictions where these errors occur. We will calculate the percentage error for each item and then examine the characteristics of the items with the largest errors.

In [ ]:
print("Calculating percentage errors for each prediction...")

# Ensure predictions_all_sphericity_fe and actuals_all_sphericity_fe are available
# If this cell is run independently, ensure the preceding cells that generate these are run first

percentage_errors_all_items = {}

for dim in ['length_cm', 'width_cm', 'height_cm']:
    predicted = predictions_all_sphericity_fe[dim]
    actual = actuals_all_sphericity_fe[dim]

    # Calculate percentage error
    # Handle cases where actual is zero to avoid division by zero, though unlikely for physical dimensions.
    current_percentage_error = np.abs((actual - predicted) / actual) * 100
    current_percentage_error = current_percentage_error.replace([np.inf, -np.inf], np.nan)
    percentage_errors_all_items[dim] = current_percentage_error

# Add these percentage errors back to a copy of df_full for analysis
df_analysis = df_full.copy()
df_analysis['predicted_length_cm'] = predictions_all_sphericity_fe['length_cm']
df_analysis['predicted_width_cm'] = predictions_all_sphericity_fe['width_cm']
df_analysis['predicted_height_cm'] = predictions_all_sphericity_fe['height_cm']

df_analysis['percentage_error_length_cm'] = percentage_errors_all_items['length_cm']
df_analysis['percentage_error_width_cm'] = percentage_errors_all_items['width_cm']
df_analysis['percentage_error_height_cm'] = percentage_errors_all_items['height_cm']

print("Percentage errors calculated and added to analysis DataFrame.")

Calculating percentage errors for each prediction...
Percentage errors calculated and added to analysis DataFrame.


In [ ]:
print("Displaying top 5 items with highest percentage errors for each dimension...")

for dim in ['length_cm', 'width_cm', 'height_cm']:
    print(f"\n--- Top 5 Highest Percentage Errors for {dim} ---")
    # Sort by percentage error in descending order, drop NaNs, and get the top 5
    top_errors = df_analysis.sort_values(by=f'percentage_error_{dim}', ascending=False).dropna(subset=[f'percentage_error_{dim}']).head(5)

    # Display relevant columns for analysis
    display(top_errors[[
        'title', dim, f'predicted_{dim}', f'percentage_error_{dim}',
        'price', 'dim_max_cm', 'dim_min_cm', 'dim_ratio',
        'total_surface_area_cm2', 'ratio_min_to_max_dim', 'mean_three_dims_cm',
        'length_width_aspect_ratio', 'length_height_aspect_ratio', 'width_height_aspect_ratio', 'sphericity'
    ]])

Displaying top 5 items with highest percentage errors for each dimension...

--- Top 5 Highest Percentage Errors for length_cm ---


,title,length_cm,predicted_length_cm,percentage_error_length_cm,price,dim_max_cm,dim_min_cm,dim_ratio,total_surface_area_cm2,ratio_min_to_max_dim,mean_three_dims_cm,length_width_aspect_ratio,length_height_aspect_ratio,width_height_aspect_ratio,sphericity
2050,Water Wheel Timer,1.27,2.243653,76.665576,8.24,1.27,1.27,1.000000,9.6774,1.000000,1.27,1.0,1.000000,1.000000,1.000000
30545,"Koplow Games Assorted Foam Spot Dice, 16mm",1.60,2.243653,40.228301,7.49,1.60,1.60,1.000000,15.3600,1.000000,1.60,1.0,1.000000,1.000000,1.000000
16804,PET MUNCHIES Salmon Bites 90g,3.00,3.679829,22.660979,38.08,3.00,3.00,1.000000,54.0000,1.000000,3.00,1.0,1.000000,1.000000,1.000000
23349,VOVIGGOL 30Pcs 1.3 Inch Cat Toy Balls Soft Kit...,3.00,3.679829,22.660979,6.99,3.00,3.00,1.000000,54.0000,1.000000,3.00,1.0,1.000000,1.000000,1.000000
1181,"Heart and Flower Badge Reel,Retractable Name C...",2.92,3.507640,20.124645,9.99,2.92,0.91,3.208791,27.6816,0.311644,2.25,1.0,3.208791,3.208791,0.879873



--- Top 5 Highest Percentage Errors for width_cm ---


,title,width_cm,predicted_width_cm,percentage_error_width_cm,price,dim_max_cm,dim_min_cm,dim_ratio,total_surface_area_cm2,ratio_min_to_max_dim,mean_three_dims_cm,length_width_aspect_ratio,length_height_aspect_ratio,width_height_aspect_ratio,sphericity
17232,PUPTECK 2 PCS Soft Leather Cat Kitten Collar-O...,0.89,1.175905,32.124116,9.99,26.92,0.51,52.784314,76.2838,0.018945,9.440000,30.247191,52.784314,1.745098,0.243991
39129,Classic Leather Dog Collar for Small Medium La...,1.50,1.958254,30.550297,12.99,37.01,0.51,72.568627,150.3102,0.013780,13.006667,24.673333,72.568627,2.941176,0.234330
6129,Outdoor Playhouse for Kids Wooden Backyard Pla...,154.94,116.038528,25.107443,229.89,162.56,116.84,1.391304,124567.4928,0.718750,144.780000,1.049180,1.391304,1.326087,0.989801
26304,Stiive Fitness Tracker with Heart Rate Monitor...,1.70,2.098226,23.425035,26.99,3.99,0.99,4.030303,24.8322,0.248120,2.226667,2.347059,4.030303,1.717172,0.847287
38801,"4-Pack Auto Hauler 13""-16"" Wheel Bonnet Ratche...",5.08,4.154521,18.218100,105.99,167.64,2.54,66.000000,2580.6400,0.015152,58.420000,33.000000,66.000000,2.000000,0.221376



--- Top 5 Highest Percentage Errors for height_cm ---


,title,height_cm,predicted_height_cm,percentage_error_height_cm,price,dim_max_cm,dim_min_cm,dim_ratio,total_surface_area_cm2,ratio_min_to_max_dim,mean_three_dims_cm,length_width_aspect_ratio,length_height_aspect_ratio,width_height_aspect_ratio,sphericity
10356,Hetoy Ride on Car for Kids 12V Power Battery E...,106.68,63.130535,40.822521,185.99,106.68,106.68,1.000000,68283.7344,1.000000,106.680000,1.000000,1.000000,1.000000,1.000000
6129,Outdoor Playhouse for Kids Wooden Backyard Pla...,116.84,69.619286,40.414853,229.89,162.56,116.84,1.391304,124567.4928,0.718750,144.780000,1.049180,1.391304,1.326087,0.989801
35199,Artcraft Lighting JA14028BK Transitional Eight...,101.60,63.130535,37.863647,570.61,116.84,101.60,1.150000,68128.8960,0.869565,106.680000,1.150000,1.150000,1.000000,0.997800
27022,SOFTSEA Upholstered Platform Bed Frame with Dr...,100.08,64.635223,35.416443,223.90,193.04,100.08,1.928857,108923.2000,0.518442,137.670000,1.610143,1.928857,1.197942,0.961066
28030,Disney Princess Indoor Playhouse with Fabric T...,107.95,71.068939,34.164947,78.39,121.28,107.95,1.123483,75675.1090,0.890089,112.393333,1.123483,1.123483,1.000000,0.998476


#### Observations from High Error Items

After examining the items with the highest percentage errors, we can look for patterns or anomalies that might explain the poor predictions. Common reasons for high errors include:

1.  **Outliers in Actual Dimensions:** Very small or very large actual dimensions compared to the rest of the dataset can be hard for the model to learn.
2.  **Unusual Aspect Ratios or Sphericity:** Products with highly irregular shapes (e.g., extremely long and thin, or very flat) might have feature combinations that are rare in the training data.
3.  **Data Entry Errors:** Mistakes in the original `length_cm`, `width_cm`, `height_cm`, or feature values could directly lead to large errors.
4.  **Limited Feature Representation:** For some extreme cases, the current set of features might not adequately capture the complexity of the product's shape or characteristics.
5.  **Small Actual Values Leading to Large Percentage Errors:** A small absolute error can translate to a very large percentage error if the actual dimension is also very small. For example, an actual `length_cm` of 1 with a prediction of 2 gives a 100% error, even if the absolute difference is only 1 cm.

**Next Steps for Investigation:**

*   **Manual Inspection of Specific Items:** For the top error items, manually check their product titles or descriptions (if available outside this dataset) to see if they correspond to unusual product types or dimensions.
*   **Distribution Analysis of Problematic Features:** Visualize the distributions of `length_cm`, `width_cm`, `height_cm`, and engineered features like aspect ratios and sphericity, especially focusing on the tails of the distributions where these high-error items might reside.
*   **Thresholding Small Values:** Consider if a different error metric or approach is needed for very small dimensions, as percentage error can be misleading there.
*   **Robustness to Outliers:** Investigate if the model (XGBoost) is overly sensitive to outliers, or if more robust preprocessing (e.g., winsorization, different scaling) could help.

### Exploring Alternative Error Metrics for Small Dimensions

As observed, very small actual dimensions can lead to disproportionately high percentage errors. To get a more robust understanding of prediction accuracy for these items, we can explore alternative metrics:

1.  **Mean Absolute Error (MAE):** This measures the average absolute difference between predicted and actual values. It's in the same units as the dimensions and isn't skewed by small actual values.
2.  **Scaled Absolute Error:** This custom metric aims to balance absolute and percentage errors. It can be defined as `Absolute Error / (Actual Value + constant)`. Adding a small constant to the denominator prevents division by zero or extremely large values when the actual dimension is very small, making it more stable than standard percentage error. We can also consider a 'clipped' percentage error, which caps the maximum percentage error to a reasonable value (e.g., 100% or 200%) to prevent extreme outliers from dominating analysis.

In [ ]:
print("Calculating alternative error metrics for top error items...")

alternative_error_analysis = {}

# Define a small constant for scaled absolute error to prevent division by near-zero values
SCALED_ERROR_CONSTANT = 0.5 # A small value in cm

for dim in ['length_cm', 'width_cm', 'height_cm']:
    print(f"\n--- Alternative Error Metrics for {dim} ---")

    # Get the top error items previously identified
    top_errors_df = df_analysis.sort_values(by=f'percentage_error_{dim}', ascending=False).dropna(subset=[f'percentage_error_{dim}']).head(5).copy()

    # Calculate Absolute Error
    top_errors_df[f'absolute_error_{dim}'] = np.abs(top_errors_df[dim] - top_errors_df[f'predicted_{dim}'])

    # Calculate Scaled Absolute Error
    # This version is a percentage-like metric but less sensitive to tiny actuals
    top_errors_df[f'scaled_absolute_error_{dim}'] = (top_errors_df[f'absolute_error_{dim}'] / (top_errors_df[dim] + SCALED_ERROR_CONSTANT)) * 100

    # Display relevant columns including new error metrics
    display(top_errors_df[[
        'title', dim, f'predicted_{dim}', f'percentage_error_{dim}',
        f'absolute_error_{dim}', f'scaled_absolute_error_{dim}',
        'price', 'dim_max_cm', 'dim_min_cm', 'sphericity'
    ]])

print("Alternative error metrics calculated and displayed.")

Calculating alternative error metrics for top error items...

--- Alternative Error Metrics for length_cm ---


,title,length_cm,predicted_length_cm,percentage_error_length_cm,absolute_error_length_cm,scaled_absolute_error_length_cm,price,dim_max_cm,dim_min_cm,sphericity
2050,Water Wheel Timer,1.27,2.243653,76.665576,0.973653,55.008634,8.24,1.27,1.27,1.000000
30545,"Koplow Games Assorted Foam Spot Dice, 16mm",1.60,2.243653,40.228301,0.643653,30.650134,7.49,1.60,1.60,1.000000
16804,PET MUNCHIES Salmon Bites 90g,3.00,3.679829,22.660979,0.679829,19.423696,38.08,3.00,3.00,1.000000
23349,VOVIGGOL 30Pcs 1.3 Inch Cat Toy Balls Soft Kit...,3.00,3.679829,22.660979,0.679829,19.423696,6.99,3.00,3.00,1.000000
1181,"Heart and Flower Badge Reel,Retractable Name C...",2.92,3.507640,20.124645,0.587640,17.182446,9.99,2.92,0.91,0.879873



--- Alternative Error Metrics for width_cm ---


,title,width_cm,predicted_width_cm,percentage_error_width_cm,absolute_error_width_cm,scaled_absolute_error_width_cm,price,dim_max_cm,dim_min_cm,sphericity
17232,PUPTECK 2 PCS Soft Leather Cat Kitten Collar-O...,0.89,1.175905,32.124116,0.285905,20.568679,9.99,26.92,0.51,0.243991
39129,Classic Leather Dog Collar for Small Medium La...,1.50,1.958254,30.550297,0.458254,22.912723,12.99,37.01,0.51,0.234330
6129,Outdoor Playhouse for Kids Wooden Backyard Pla...,154.94,116.038528,25.107443,38.901472,25.026680,229.89,162.56,116.84,0.989801
26304,Stiive Fitness Tracker with Heart Rate Monitor...,1.70,2.098226,23.425035,0.398226,18.101163,26.99,3.99,0.99,0.847287
38801,"4-Pack Auto Hauler 13""-16"" Wheel Bonnet Ratche...",5.08,4.154521,18.218100,0.925479,16.585654,105.99,167.64,2.54,0.221376



--- Alternative Error Metrics for height_cm ---


,title,height_cm,predicted_height_cm,percentage_error_height_cm,absolute_error_height_cm,scaled_absolute_error_height_cm,price,dim_max_cm,dim_min_cm,sphericity
10356,Hetoy Ride on Car for Kids 12V Power Battery E...,106.68,63.130535,40.822521,43.549465,40.632081,185.99,106.68,106.68,1.000000
6129,Outdoor Playhouse for Kids Wooden Backyard Pla...,116.84,69.619286,40.414853,47.220714,40.242641,229.89,162.56,116.84,0.989801
35199,Artcraft Lighting JA14028BK Transitional Eight...,101.60,63.130535,37.863647,38.469465,37.678222,570.61,116.84,101.60,0.997800
27022,SOFTSEA Upholstered Platform Bed Frame with Dr...,100.08,64.635223,35.416443,35.444777,35.240382,223.90,193.04,100.08,0.961066
28030,Disney Princess Indoor Playhouse with Fabric T...,107.95,71.068939,34.164947,36.881061,34.007433,78.39,121.28,107.95,0.998476


Alternative error metrics calculated and displayed.


### Filtering Items with Absolute Error Below 1 cm

To further understand the model's performance on smaller items, let's filter the `df_analysis` DataFrame to include only those products where the absolute prediction error for each dimension (`length_cm`, `width_cm`, `height_cm`) is less than 1 cm. Then we will re-examine their percentage errors.

In [ ]:
print("Filtering items with absolute error below 1 cm for each dimension...")

# Calculate absolute errors for all items and add them to df_analysis
for dim in ['length_cm', 'width_cm', 'height_cm']:
    df_analysis[f'absolute_error_{dim}'] = np.abs(df_analysis[dim] - df_analysis[f'predicted_{dim}'])

# Filter df_analysis for items where absolute error is less than 1 cm for each dimension
# We'll consider an item 'filtered' if its absolute error is < 1cm for the *specific dimension being analyzed*.
# This means the top_errors_df will be created for each dimension based on its own absolute error.

print("Displaying top 5 percentage errors for items where absolute error is less than 1 cm...")

for dim in ['length_cm', 'width_cm', 'height_cm']:
    print(f"\n--- Top 5 Percentage Errors for {dim} (Absolute Error < 1 cm) ---")

    # Filter the DataFrame for the current dimension where absolute error is less than 1 cm
    filtered_df_for_dim = df_analysis[df_analysis[f'absolute_error_{dim}'] < 1].copy()

    if filtered_df_for_dim.empty:
        print(f"No items found with absolute error less than 1 cm for {dim}.")
        continue

    # Sort by percentage error in descending order, drop NaNs, and get the top 5 from the filtered set
    top_errors_filtered = filtered_df_for_dim.sort_values(
        by=f'percentage_error_{dim}', ascending=False
    ).dropna(subset=[f'percentage_error_{dim}']).head(5)

    # Display relevant columns for analysis
    display(top_errors_filtered[[
        'title', dim, f'predicted_{dim}', f'percentage_error_{dim}',
        f'absolute_error_{dim}', 'price', 'dim_max_cm', 'dim_min_cm',
        'sphericity'
    ]])

print("Filtering and display of results complete.")

Filtering items with absolute error below 1 cm for each dimension...
Displaying top 5 percentage errors for items where absolute error is less than 1 cm...

--- Top 5 Percentage Errors for length_cm (Absolute Error < 1 cm) ---


,title,length_cm,predicted_length_cm,percentage_error_length_cm,absolute_error_length_cm,price,dim_max_cm,dim_min_cm,sphericity
2050,Water Wheel Timer,1.27,2.243653,76.665576,0.973653,8.24,1.27,1.27,1.000000
30545,"Koplow Games Assorted Foam Spot Dice, 16mm",1.60,2.243653,40.228301,0.643653,7.49,1.60,1.60,1.000000
16804,PET MUNCHIES Salmon Bites 90g,3.00,3.679829,22.660979,0.679829,38.08,3.00,3.00,1.000000
23349,VOVIGGOL 30Pcs 1.3 Inch Cat Toy Balls Soft Kit...,3.00,3.679829,22.660979,0.679829,6.99,3.00,3.00,1.000000
1181,"Heart and Flower Badge Reel,Retractable Name C...",2.92,3.507640,20.124645,0.587640,9.99,2.92,0.91,0.879873



--- Top 5 Percentage Errors for width_cm (Absolute Error < 1 cm) ---


,title,width_cm,predicted_width_cm,percentage_error_width_cm,absolute_error_width_cm,price,dim_max_cm,dim_min_cm,sphericity
17232,PUPTECK 2 PCS Soft Leather Cat Kitten Collar-O...,0.89,1.175905,32.124116,0.285905,9.99,26.92,0.51,0.243991
39129,Classic Leather Dog Collar for Small Medium La...,1.50,1.958254,30.550297,0.458254,12.99,37.01,0.51,0.234330
26304,Stiive Fitness Tracker with Heart Rate Monitor...,1.70,2.098226,23.425035,0.398226,26.99,3.99,0.99,0.847287
38801,"4-Pack Auto Hauler 13""-16"" Wheel Bonnet Ratche...",5.08,4.154521,18.218100,0.925479,105.99,167.64,2.54,0.221376
17773,Odoria 1/12 Miniature Baseball Bat Dollhouse A...,0.71,0.824236,16.089540,0.114236,3.99,6.30,0.71,0.571202



--- Top 5 Percentage Errors for height_cm (Absolute Error < 1 cm) ---


,title,height_cm,predicted_height_cm,percentage_error_height_cm,absolute_error_height_cm,price,dim_max_cm,dim_min_cm,sphericity
2220,Aluminum 11inch Heat Diffuser Plate For Gas St...,0.61,0.544231,10.781745,0.065769,32.80,27.94,0.61,0.414709
3362,The University of Akron UA Zips Sticker Vinyl ...,0.61,0.544231,10.781745,0.065769,10.99,34.01,0.61,0.400076
21567,Big Dot of Happiness Silver Tassel Worth The H...,0.61,0.544231,10.781745,0.065769,9.99,32.59,0.61,0.409827
33378,"SJT ENTERPRISES, INC. Grandparent House Rules:...",0.61,0.544231,10.781745,0.065769,19.99,41.00,0.61,0.385130
12600,Robert Morris University Sticker Colonials RMU...,0.61,0.544231,10.781745,0.065769,11.99,34.01,0.61,0.400076


Filtering and display of results complete.


In [ ]:
# não elimine esse código
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

print("\n--- Calculating Relative MAE on Original Scale (XGBoost with Newly Engineered Sphericity Feature) ---")

relative_mae_results_sphericity_fe = {}

# Use the target_cols_renamed which now includes dimensions, as set in c07cc6ae
# Ensure it's globally accessible for consistency

for col in target_cols_renamed:
    # Get the trained model for the current target
    model = re_trained_optimized_models_sphericity_fe[col]

    # Make predictions on the full test set (log-transformed)
    y_pred_log = model.predict(X_test_full)

    # Get actual values from the full test set (log-transformed)
    y_actual_log = y_test_full[col]

    # Inverse transform predictions and actual values to original scale
    y_pred_original = np.expm1(y_pred_log)
    y_actual_original = np.expm1(y_actual_log)

    # Calculate MAE on the original scale
    mae_original_scale = mean_absolute_error(y_actual_original, y_pred_original)

    # Calculate the average actual value on the original scale
    avg_actual_original = y_actual_original.mean()

    # Calculate relative MAE
    if avg_actual_original > 0:
        relative_mae = (mae_original_scale / avg_actual_original) * 100
    else:
        relative_mae = np.nan # Avoid division by zero

    original_col_name = col.replace('log_', '') # Get original column name (e.g., weight_g)
    relative_mae_results_sphericity_fe[original_col_name] = {
        'MAE_original_scale': mae_original_scale,
        'Avg_Actual_original_scale': avg_actual_original,
        'Relative_MAE_%': relative_mae
    }

# Display the results
relative_mae_df_sphericity_fe = pd.DataFrame(relative_mae_results_sphericity_fe).T
print("\nRelative Mean Absolute Error (MAE) on Original Scale (XGBoost with Newly Engineered Sphericity Feature):")
display(relative_mae_df_sphericity_fe)

# Compare with previous XGBoost results (XGBoost with previous new features)
print("\n--- Comparison of Relative MAE with Previous XGBoost Results (with previous new features) ---")
# Assuming relative_mae_df_new_fe was created in a prior cell and is available
# If not, this comparison will fail, or need a fallback for relative_mae_df_new_fe
if 'relative_mae_df_new_fe' in globals():
    print("XGBoost with Previous New Features Relative MAE:")
    display(relative_mae_df_new_fe)
else:
    print("Previous results (relative_mae_df_new_fe) not available for comparison.")

print("\nXGBoost with Newly Engineered Sphericity Feature Relative MAE:")
display(relative_mae_df_sphericity_fe)

# Check against the 5% target
print("\n--- Checking Relative MAE against 5% target (XGBoost with Newly Engineered Sphericity Feature) ---")
for product, metrics in relative_mae_results_sphericity_fe.items():
    if metrics['Relative_MAE_%'] <= 5:
        print(f"  {product.capitalize()} Relative MAE ({metrics['Relative_MAE_%']:.2f}%) is within the 5% target.")
    else:
        print(f"  WARNING: {product.capitalize()} Relative MAE ({metrics['Relative_MAE_%']:.2f}%) EXCEEDS the 5% target.")


--- Calculating Relative MAE on Original Scale (XGBoost with Newly Engineered Sphericity Feature) ---

Relative Mean Absolute Error (MAE) on Original Scale (XGBoost with Newly Engineered Sphericity Feature):


,MAE_original_scale,Avg_Actual_original_scale,Relative_MAE_%
length_cm,0.241358,27.284382,0.884603
width_cm,0.335599,17.005374,1.973489
height_cm,0.119575,7.230963,1.653655



--- Comparison of Relative MAE with Previous XGBoost Results (with previous new features) ---
Previous results (relative_mae_df_new_fe) not available for comparison.

XGBoost with Newly Engineered Sphericity Feature Relative MAE:


,MAE_original_scale,Avg_Actual_original_scale,Relative_MAE_%
length_cm,0.241358,27.284382,0.884603
width_cm,0.335599,17.005374,1.973489
height_cm,0.119575,7.230963,1.653655



--- Checking Relative MAE against 5% target (XGBoost with Newly Engineered Sphericity Feature) ---
  Length_cm Relative MAE (0.88%) is within the 5% target.
  Width_cm Relative MAE (1.97%) is within the 5% target.
  Height_cm Relative MAE (1.65%) is within the 5% target.
